In [0]:
# Load Dataset
df = spark.read.csv(
    "/Volumes/workspace/default/Volume/Transactions.csv",
    header=True,
    inferSchema=True
)

display(df)


transaction_id,customer_id,transaction_date,amount,status,country
100001,6050,2020-12-03,11882.92,Completed,Canada
100002,3216,2024-05-05,9513.66,Active,Canada
100003,9645,2019-02-04,14640.97,Completed,India
100004,5775,2024-08-07,7937.4,Closed,Australia
100005,4816,2022-02-12,11986.44,Closed,UK
100006,5234,2018-08-22,9264.21,Active,Canada
100007,5560,2018-01-10,11508.34,Pending,Canada
100008,8377,2026-01-05,6719.36,Active,USA
100009,1206,2022-07-20,5841.95,Pending,USA
100010,4061,2025-11-09,13881.72,Pending,Canada


In [0]:
# Create Active Dataset
# Active Data
# = transactions from last 12 months
from pyspark.sql.functions import col, current_date, datediff

active_data = df.filter(
    datediff(current_date(), col("transaction_date")) <= 365
)

display(active_data)


transaction_id,customer_id,transaction_date,amount,status,country
100008,8377,2026-01-05,6719.36,Active,USA
100010,4061,2025-11-09,13881.72,Pending,Canada
100011,4483,2025-05-31,1131.19,Pending,UK
100015,2291,2025-06-14,1995.15,Pending,Australia
100026,8558,2025-06-08,14722.3,Closed,Australia
100033,7334,2025-06-22,7179.11,Completed,Australia
100043,5398,2025-10-20,5347.96,Active,Australia
100053,8337,2025-12-18,5458.94,Closed,Australia
100054,3418,2026-02-18,4332.04,Active,Canada
100056,7781,2025-07-21,14657.64,Pending,India


In [0]:
# Create Historical Dataset
# Assume:
# Historical Data
# = older than 12 months
historical_data = df.filter(
    datediff(current_date(), col("transaction_date")) > 365
)

display(historical_data)


transaction_id,customer_id,transaction_date,amount,status,country
100001,6050,2020-12-03,11882.92,Completed,Canada
100002,3216,2024-05-05,9513.66,Active,Canada
100003,9645,2019-02-04,14640.97,Completed,India
100004,5775,2024-08-07,7937.4,Closed,Australia
100005,4816,2022-02-12,11986.44,Closed,UK
100006,5234,2018-08-22,9264.21,Active,Canada
100007,5560,2018-01-10,11508.34,Pending,Canada
100009,1206,2022-07-20,5841.95,Pending,USA
100012,4909,2018-05-22,2350.23,Completed,UK
100013,4251,2021-02-23,7322.74,Completed,UK


# Archive Process

In [0]:
# Save Active Table
active_data.write.mode("overwrite").saveAsTable(
    "transactions_active"
)


In [0]:
# Save Archive Table
historical_data.write.mode("overwrite").saveAsTable(
    "transactions_archive"
)


# Verify Archive Movement
```
SELECT COUNT(*) FROM transactions_active;
```
```
SELECT COUNT(*) FROM transactions_archive;
```

# Deletion Workflow

In [0]:
# Identify Expired Data
expired_data = df.filter(
    datediff(current_date(), col("transaction_date")) > 2555
)

display(expired_data)


transaction_id,customer_id,transaction_date,amount,status,country
100003,9645,2019-02-04,14640.97,Completed,India
100006,5234,2018-08-22,9264.21,Active,Canada
100007,5560,2018-01-10,11508.34,Pending,Canada
100012,4909,2018-05-22,2350.23,Completed,UK
100018,9771,2019-02-19,14834.21,Active,USA
100019,8271,2018-12-06,9255.99,Closed,Canada
100027,3022,2018-08-05,8747.71,Closed,USA
100029,4456,2017-02-26,14408.29,Active,UK
100041,6527,2017-12-20,154.87,Pending,UK
100046,1607,2018-05-10,12265.02,Active,USA


In [0]:
# Save as Deleted Table
expired_data.write \
    .mode("overwrite")   \
    .saveAsTable("expired_data")


In [0]:
# Simulate Delete Process
retained_data = df.filter(
    datediff(current_date(), col("transaction_date")) <= 2555
)

display(retained_data)


transaction_id,customer_id,transaction_date,amount,status,country
100001,6050,2020-12-03,11882.92,Completed,Canada
100002,3216,2024-05-05,9513.66,Active,Canada
100004,5775,2024-08-07,7937.4,Closed,Australia
100005,4816,2022-02-12,11986.44,Closed,UK
100008,8377,2026-01-05,6719.36,Active,USA
100009,1206,2022-07-20,5841.95,Pending,USA
100010,4061,2025-11-09,13881.72,Pending,Canada
100011,4483,2025-05-31,1131.19,Pending,UK
100013,4251,2021-02-23,7322.74,Completed,UK
100014,2162,2020-12-10,5861.19,Pending,Australia


# Delta Version Tracking

In [0]:
# Save as Delta Table
active_data.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("transactions_delta")


# Lifecycle Automation Script

In [0]:

# Create Simple Archive Script
import shutil
import os
from datetime import datetime

source = "/Volumes/workspace/default/Volume/Transactions.csv"
archive = "/Volumes/workspace/default/Volume/archive/Transactions.csv"

os.makedirs("/Volumes/workspace/default/Volume/archive/", exist_ok=True)

shutil.copy(source, archive)

print("Archive completed:", datetime.now())


Archive completed: 2026-04-29 17:46:17.023459


In [0]:
# Log Actions
log = {
    "action": "archive",
    "file": "Transactions.csv",
    "timestamp": str(datetime.now())
}

print(log)


{'action': 'archive', 'file': 'Transactions.csv', 'timestamp': '2026-04-29 17:46:35.180346'}
